<a href="https://colab.research.google.com/github/dophanthanhdat/project-3/blob/main/Appdoanhthu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# APP DỰ ĐOÁN DOANH THU QUÁN CAFE / TRÀ SỮA
# ============================================================

import os
import sys
import pandas as pd
import numpy as np
import customtkinter as ctk
from tkinter import messagebox
import tkinter as tk

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# ĐỌC FILE EXCEL
# ============================================================

FILE_PATH = r"D:\project3\doanhthu.xlsx.xlsx"

try:
    df = pd.read_excel(FILE_PATH)
except FileNotFoundError:
    tk.messagebox.showerror(
        "Lỗi",
        f"Không tìm thấy file:\n{FILE_PATH}"
    )
    sys.exit()

# ============================================================
# FEATURES & TARGET
# ============================================================

FEATURES = [
    "gia_tb",
    "tong_khach_ngay_thuong",
    "vi_tri_score",
    "doi_thu_num",
    "so_nhan_vien_num",
    "delivery",
    "cho_ngoi_lau",
    "dien_tich_num"
]

TARGET_NGAY = "doanh_thu_ngay_thuong"
TARGET_CUOI = "doanh_thu_cuoi_tuan"
TARGET_TUAN = "doanh_thu_tuan"

# ============================================================
# TRAIN MODEL
# ============================================================

def train_model(target):

    cols = FEATURES + [target]

    data = df[cols].dropna()

    X = data[FEATURES]
    y = data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_s, y_train)

    preds = model.predict(X_test_s)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)

    return model, scaler, r2, mae


model_ngay, scaler_ngay, r2_ngay, mae_ngay = train_model(TARGET_NGAY)
model_cuoi, scaler_cuoi, r2_cuoi, mae_cuoi = train_model(TARGET_CUOI)
model_tuan, scaler_tuan, r2_tuan, mae_tuan = train_model(TARGET_TUAN)

# ============================================================
# FUZZY LOGIC
# ============================================================

def triangular_mf(x, a, b, c):

    if x <= a or x >= c:
        return 0.0

    elif x <= b:
        return (x - a) / (b - a)

    else:
        return (c - x) / (c - b)


def trapezoidal_mf(x, a, b, c, d):

    if x <= a or x >= d:
        return 0.0

    elif x <= b:
        return (x - a) / (b - a)

    elif x <= c:
        return 1.0

    else:
        return (d - x) / (d - c)


def fuzzy_adjustment(rating, service, season, weather, event):

    mu_rating_low = triangular_mf(rating, 1, 2, 3)
    mu_rating_mid = triangular_mf(rating, 2.5, 3.5, 4.5)
    mu_rating_high = trapezoidal_mf(rating, 4, 4.5, 5, 5)

    mu_service_bad = triangular_mf(service, 1, 1.5, 2.5)
    mu_service_ok = triangular_mf(service, 2, 3, 4)
    mu_service_good = trapezoidal_mf(service, 3.5, 4, 5, 5)

    season_boost = {
        1: 1.25,
        2: 1.10,
        3: 1.00,
        4: 0.85
    }

    weather_boost = {
        1: 1.05,
        2: 1.00,
        3: 0.90
    }

    event_boost = {
        1: 1.15,
        2: 1.00
    }

    rule1 = min(mu_rating_high, mu_service_good)
    rule2 = min(mu_rating_mid, mu_service_ok)
    rule3 = max(mu_rating_low, mu_service_bad)

    outputs = [1.20, 1.00, 0.75]
    strengths = [rule1, rule2, rule3]

    total = sum(strengths)

    if total == 0:
        base = 1.0
    else:
        base = sum(s * o for s, o in zip(strengths, outputs)) / total

    factor = (
        base *
        season_boost.get(season, 1.0) *
        weather_boost.get(weather, 1.0) *
        event_boost.get(event, 1.0)
    )

    factor = max(0.6, min(1.5, factor))

    return factor

# ============================================================
# GIAO DIỆN
# ============================================================

ctk.set_appearance_mode("light")
ctk.set_default_color_theme("blue")

app = ctk.CTk()

app.title("Dự Đoán Doanh Thu Quán Cafe & Trà Sữa")
app.geometry("900x950")

COLOR_HEADER = "#1a1a2e"
COLOR_ACCENT = "#e94560"
COLOR_BG = "#f4f6f9"
COLOR_CARD = "#ffffff"

# ============================================================
# HEADER
# ============================================================

header = ctk.CTkFrame(
    app,
    fg_color=COLOR_HEADER,
    corner_radius=0,
    height=90
)

header.pack(fill="x")
header.pack_propagate(False)

ctk.CTkLabel(
    header,
    text="☕ DỰ ĐOÁN DOANH THU QUÁN CAFE & TRÀ SỮA",
    font=("Arial", 24, "bold"),
    text_color="white"
).pack(pady=(15, 2))

ctk.CTkLabel(
    header,
    text="Nhập thông số quán để AI dự đoán doanh thu",
    font=("Arial", 12),
    text_color="#cccccc"
).pack()

# ============================================================
# SCROLL FRAME
# ============================================================

main_scroll = ctk.CTkScrollableFrame(
    app,
    fg_color=COLOR_BG
)

main_scroll.pack(fill="both", expand=True)

# ============================================================
# SECTION FUNCTION
# ============================================================

def make_section(parent, title):

    frame = ctk.CTkFrame(
        parent,
        fg_color=COLOR_CARD,
        corner_radius=15
    )

    frame.pack(fill="x", padx=20, pady=10)

    ctk.CTkLabel(
        frame,
        text=title,
        font=("Arial", 16, "bold"),
        text_color="#2c3e50"
    ).pack(anchor="w", padx=15, pady=(12, 5))

    line = ctk.CTkFrame(
        frame,
        fg_color=COLOR_ACCENT,
        height=3
    )

    line.pack(fill="x", padx=15, pady=(0, 10))

    return frame

# ============================================================
# SECTION 1
# ============================================================

sec1 = make_section(main_scroll, "📊 THÔNG TIN QUÁN")

entries = {}

input_fields = [

    ("💰 Giá trung bình 1 ly (VNĐ)", "gia_tb", "45000"),

    ("👥 Tổng khách/ngày thường", "tong_khach_ngay_thuong", "250"),

    ("📍 Điểm vị trí (1-5)", "vi_tri_score", "3.5"),

    ("🏪 Số đối thủ trong 500m", "doi_thu_num", "7"),

    ("👨‍🍳 Số nhân viên", "so_nhan_vien_num", "5"),

    ("📐 Diện tích quán (m²)", "dien_tich_num", "100")
]

for label_text, key, placeholder in input_fields:

    card = ctk.CTkFrame(
        sec1,
        fg_color="#f8f9fa",
        corner_radius=10,
        border_width=1,
        border_color="#dcdde1"
    )

    card.pack(fill="x", padx=15, pady=8)

    ctk.CTkLabel(
        card,
        text=label_text,
        font=("Arial", 13, "bold"),
        text_color="#2c3e50"
    ).pack(anchor="w", padx=12, pady=(8, 2))

    entry = ctk.CTkEntry(
        card,
        height=42,
        font=("Arial", 13),
        placeholder_text=placeholder,
        corner_radius=8,
        border_width=1,
        border_color="#bdc3c7",
        fg_color="white"
    )

    entry.pack(fill="x", padx=12, pady=(0, 10))

    entries[key] = entry

# ============================================================
# SWITCH
# ============================================================

switch_frame = ctk.CTkFrame(
    sec1,
    fg_color="transparent"
)

switch_frame.pack(fill="x", padx=15, pady=10)

delivery_var = ctk.BooleanVar(value=True)
stay_var = ctk.BooleanVar(value=True)

ctk.CTkLabel(
    switch_frame,
    text="🚚 Có Delivery",
    font=("Arial", 12, "bold")
).pack(side="left", padx=(0, 10))

ctk.CTkSwitch(
    switch_frame,
    text="",
    variable=delivery_var
).pack(side="left", padx=(0, 30))

ctk.CTkLabel(
    switch_frame,
    text="📚 Cho ngồi học/làm việc",
    font=("Arial", 12, "bold")
).pack(side="left", padx=(0, 10))

ctk.CTkSwitch(
    switch_frame,
    text="",
    variable=stay_var
).pack(side="left")

# ============================================================
# SECTION 2
# ============================================================

sec2 = make_section(main_scroll, "⭐ YẾU TỐ ĐỊNH TÍNH")

# Rating
rating_var = ctk.DoubleVar(value=4.2)

ctk.CTkLabel(
    sec2,
    text="⭐ Rating Google",
    font=("Arial", 12, "bold")
).pack(anchor="w", padx=15)

rating_slider = ctk.CTkSlider(
    sec2,
    from_=1,
    to=5,
    number_of_steps=40,
    variable=rating_var
)

rating_slider.pack(fill="x", padx=15, pady=10)

# Service
service_var = ctk.DoubleVar(value=3.5)

ctk.CTkLabel(
    sec2,
    text="🤝 Chất lượng phục vụ",
    font=("Arial", 12, "bold")
).pack(anchor="w", padx=15)

service_slider = ctk.CTkSlider(
    sec2,
    from_=1,
    to=5,
    number_of_steps=40,
    variable=service_var
)

service_slider.pack(fill="x", padx=15, pady=10)

# Option Menu
MUA_MAP = {
    "Tết / Lễ lớn (+25%)": 1,
    "Mùa hè (+10%)": 2,
    "Bình thường": 3,
    "Thấp điểm (-15%)": 4
}

TIET_MAP = {
    "Đẹp / Nắng (+5%)": 1,
    "Trung bình": 2,
    "Mưa / Xấu (-10%)": 3
}

SK_MAP = {
    "Có sự kiện đặc biệt (+15%)": 1,
    "Không có sự kiện": 2
}

mua_var = ctk.StringVar(value="Bình thường")
weather_var = ctk.StringVar(value="Trung bình")
event_var = ctk.StringVar(value="Không có sự kiện")

ctk.CTkOptionMenu(
    sec2,
    values=list(MUA_MAP.keys()),
    variable=mua_var,
    height=40
).pack(fill="x", padx=15, pady=8)

ctk.CTkOptionMenu(
    sec2,
    values=list(TIET_MAP.keys()),
    variable=weather_var,
    height=40
).pack(fill="x", padx=15, pady=8)

ctk.CTkOptionMenu(
    sec2,
    values=list(SK_MAP.keys()),
    variable=event_var,
    height=40
).pack(fill="x", padx=15, pady=8)

# ============================================================
# KẾT QUẢ
# ============================================================

sec3 = make_section(main_scroll, "📈 KẾT QUẢ DỰ ĐOÁN")

result_label = ctk.CTkTextbox(
    sec3,
    height=220,
    font=("Consolas", 14)
)

result_label.pack(fill="both", padx=15, pady=10)

# ============================================================
# HÀM DỰ ĐOÁN
# ============================================================

def predict_revenue():

    try:

        vals = []

        for label_text, key, _ in input_fields:

            raw = entries[key].get().strip().replace(",", "")

            if raw == "":
                messagebox.showwarning(
                    "Thiếu dữ liệu",
                    f"Vui lòng nhập:\n{label_text}"
                )
                return

            vals.append(float(raw))

        features = [

            vals[0],
            vals[1],
            vals[2],
            vals[3],
            vals[4],

            1 if delivery_var.get() else 0,

            1 if stay_var.get() else 0,

            vals[5]
        ]

        X_input = np.array([features])

        pred_ngay = model_ngay.predict(
            scaler_ngay.transform(X_input)
        )[0]

        pred_cuoi = model_cuoi.predict(
            scaler_cuoi.transform(X_input)
        )[0]

        pred_tuan = model_tuan.predict(
            scaler_tuan.transform(X_input)
        )[0]

        factor = fuzzy_adjustment(
            rating_var.get(),
            service_var.get(),
            MUA_MAP[mua_var.get()],
            TIET_MAP[weather_var.get()],
            SK_MAP[event_var.get()]
        )

        final_ngay = pred_ngay * factor
        final_cuoi = pred_cuoi * factor
        final_tuan = pred_tuan * factor

        final_thang = final_tuan * 4.33

        result_text = f"""
================ KẾT QUẢ =================

📅 Doanh thu ngày thường:
{final_ngay:,.0f} VNĐ

🎉 Doanh thu cuối tuần:
{final_cuoi:,.0f} VNĐ

📆 Doanh thu tháng:
{final_thang:,.0f} VNĐ

------------------------------------------

🤖 Hệ số AI điều chỉnh:
{factor:.2f}

📊 Độ chính xác mô hình:

Ngày thường: {r2_ngay*100:.1f}%
Cuối tuần: {r2_cuoi*100:.1f}%
Theo tuần: {r2_tuan*100:.1f}%

==========================================
"""

        result_label.delete("1.0", "end")
        result_label.insert("end", result_text)

    except ValueError:
        messagebox.showerror(
            "Lỗi",
            "Vui lòng nhập đúng dữ liệu số!"
        )

    except Exception as e:
        messagebox.showerror(
            "Lỗi",
            str(e)
        )

# ============================================================
# RESET
# ============================================================

def reset_inputs():

    for _, key, _ in input_fields:
        entries[key].delete(0, "end")

    result_label.delete("1.0", "end")

# ============================================================
# BUTTON
# ============================================================

btn_frame = ctk.CTkFrame(
    main_scroll,
    fg_color="transparent"
)

btn_frame.pack(pady=15)

predict_btn = ctk.CTkButton(
    btn_frame,
    text="🔍 DỰ ĐOÁN DOANH THU",
    font=("Arial", 16, "bold"),
    width=320,
    height=55,
    fg_color=COLOR_ACCENT,
    hover_color="#c0392b",
    command=predict_revenue
)

predict_btn.pack(pady=5)

reset_btn = ctk.CTkButton(
    btn_frame,
    text="↺ NHẬP LẠI",
    font=("Arial", 14),
    width=200,
    height=45,
    fg_color="#7f8c8d",
    hover_color="#636e72",
    command=reset_inputs
)

reset_btn.pack(pady=5)

# ============================================================
# FOOTER
# ============================================================

footer = ctk.CTkFrame(
    app,
    fg_color=COLOR_HEADER,
    corner_radius=0,
    height=35
)

footer.pack(fill="x", side="bottom")
footer.pack_propagate(False)

ctk.CTkLabel(
    footer,
    text=f"Dữ liệu: {len(df)} quán | AI Revenue Prediction System",
    font=("Arial", 10),
    text_color="#cccccc"
).pack(expand=True)

# ============================================================
# RUN APP
# ============================================================

app.mainloop()